In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Connect and Access Git Repository

import sys
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"



token = getpass("GitHub token: ")


REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists(): 
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())


In [ ]:
# Imports
import os
import numpy as np
import torch




# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.seed import set_seed, RANDOM_SEED
from models.anoformer import AnoFormerMainModel
from explainability.anomaly_score_wrappers import AnoFormerAnomalyScoreWrapper


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

In [ ]:
# Paths

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models"
MODEL_PATH = MODEL_DIR / "anoformer_40epochs_larger_capacity.pt"

SHAP_RESULTS_DIR = RESULTS_DIR / "anoformer" / "shap"
SHAP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


print("MODEL_PATH:", MODEL_PATH)
print("SHAP results:", SHAP_RESULTS_DIR)

In [ ]:
# Load my preprocessed data

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_val = np.load(PROCESSED_DIR / "X_val.npy")
y_val = np.load(PROCESSED_DIR / "y_val.npy")

X_test = np.load(PROCESSED_DIR / "X_test.npy")
y_test = np.load(PROCESSED_DIR / "y_test.npy")

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

In [ ]:
AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

In [ ]:
# Load the pre-trained model

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}"
    )

AnoFormer.Load(str(MODEL_PATH))
AnoFormer.Generator.to(DEVICE)
AnoFormer.Generator.eval()

print("AnoFormer checkpoint loaded.")

In [ ]:
# Create the Score Wrapper

ScoreWrapper = AnoFormerAnomalyScoreWrapper(
    anoformer_model=AnoFormer,
    window_size=1000,
    masking_rate=None,  # Uses the rate stored in AnoFormer.Masking
    num_masks=1, #try 3 later
    seed=RANDOM_SEED,
).to(DEVICE)

ScoreWrapper.eval()

In [ ]:
# Select Background ECG samples from training set for SHAP

BACKGROUND_SIZE = 20

normal_train_indices = np.where(y_train == 0)[0]

rng = np.random.default_rng(RANDOM_SEED)

background_indices = rng.choice(
    normal_train_indices,
    size=BACKGROUND_SIZE,
    replace=False,
)

background_ecgs = torch.as_tensor(
    X_train[background_indices],
    dtype=torch.float32,
    device=DEVICE,
)

print("Background shape:", background_ecgs.shape)
print("Background labels:", y_train[background_indices])

In [ ]:
# Save the background indices

np.save(
    SHAP_RESULTS_DIR / "background_indices.npy",
    background_indices,
)